In [1]:
import os
import warnings

import joblib
import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")

# ============================================================
# 1. LOAD & CLEAN DATA
# ============================================================

In [2]:
df=pd.read_csv("pha_dataset_cleaned.csv")

In [3]:
# Regression features
feature_cols = [
    "a",
    "e",
    "i",
    "q",
    "ad",
    "n",
    "per",
    "H",
    "om",
    "w",
    "ma",
    "class",
]

# Target
target_col = "diameter"

# ------------------------------------------------------------
# Clean target column
# ------------------------------------------------------------


In [4]:
# Convert diameter to numeric
df[target_col] = pd.to_numeric(df[target_col],errors="coerce")

# Remove missing diameter
df_clean = df.dropna(subset=[target_col]).copy()

# Remove zero and negative diameter
df_clean = df_clean[df_clean[target_col] > 0]


# ------------------------------------------------------------
# Physical sanity check
# ------------------------------------------------------------


In [5]:
df_clean = df_clean[(df_clean["e"] >= 0)  & (df_clean["e"] < 1.0)  & (df_clean["a"] > 0)]

In [6]:
# Create X and y
# ------------------------------------------------------------

X = df_clean[feature_cols].copy()

y = df_clean[target_col].copy()


print("\nAfter cleaning:")
print("X shape:", X.shape)
print("y shape:", y.shape)


After cleaning:
X shape: (136209, 12)
y shape: (136209,)


In [8]:
# 2. LOG TRANSFORMATION OF TARGET
# ============================================================

# Diameter is right-skewed.
# log1p() reduces the effect of extreme large values.

y_log = np.log1p(y)

In [9]:
# 3. TRAIN-TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_log,
    test_size=0.20,
    random_state=42)

print("\nAfter train-test split:")
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)


After train-test split:
X_train: (108967, 12)
X_test : (27242, 12)
y_train: (108967,)
y_test : (27242,)


In [10]:
# 4. MISSING VALUE IMPUTATION
# ============================================================

num_cols = [
    "a",
    "e",
    "i",
    "q",
    "ad",
    "n",
    "per",
    "H",
    "om",
    "w",
    "ma",]

In [11]:
# Create imputer
imputer = SimpleImputer(strategy="median")


# Fit only on training data
X_train[num_cols] = imputer.fit_transform(X_train[num_cols])

# Apply same imputer to test data
X_test[num_cols] = imputer.transform(X_test[num_cols])

print("\nMissing values handled.")


Missing values handled.


In [12]:
# 5. LOG TRANSFORMATION OF SKEWED FEATURES
# ============================================================

skewed_cols = [ "a","q","ad","per","n",]

for col in skewed_cols:

    # Train
    X_train[f"log_{col}"] = np.log1p(X_train[col].clip(lower=0)    )

    # Test
    X_test[f"log_{col}"] = np.log1p(X_test[col].clip(lower=0))

    # Remove original column
    X_train.drop(columns=[col],inplace=True )

    X_test.drop(columns=[col],inplace=True )

print("\nSkewed numerical features transformed.")


Skewed numerical features transformed.


In [13]:
# 6. HANDLE CATEGORICAL COLUMN
# ============================================================

# Fill missing class values
X_train["class"] = X_train["class"].fillna("Unknown")

X_test["class"] = X_test["class"].fillna("Unknown")


In [ ]:
# 7. ONE-HOT ENCODING
# ============================================================

ohe = OneHotEncoder(
    sparse_output=False,
    handle_unknown="ignore")


# Fit encoder on training data
train_encoded = ohe.fit_transform(X_train[["class"]])
# Transform test data
test_encoded = ohe.transform(X_test[["class"]])

In [ ]:
# Convert encoded arrays into DataFrames
train_encoded_df = pd.DataFrame(
    train_encoded,
    columns=ohe.get_feature_names_out(["class"]),
    index=X_train.index)

test_encoded_df = pd.DataFrame(
    test_encoded,
    columns=ohe.get_feature_names_out(["class"]),
    index=X_test.index)

In [16]:
# Combine numerical + encoded categorical features
X_train = pd.concat([
        X_train.drop(columns=["class"]),
        train_encoded_df ],axis=1)

X_test = pd.concat([
        X_test.drop(columns=["class"]),
        test_encoded_df ], axis=1)


print("\nOne-Hot Encoding completed.")
print("Number of features:", X_train.shape[1])


One-Hot Encoding completed.
Number of features: 22


In [19]:
# 8. FEATURE SCALING
# ============================================================

scaler = StandardScaler()


# Fit scaler only on training data
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index)


# Apply same scaler to test data
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index)

print("\nFeature scaling completed.")


Feature scaling completed.


In [24]:
import os

# Create folder if it does not exist
os.makedirs("processed_regression", exist_ok=True)

print("Folder created successfully!")

Folder created successfully!


In [25]:
# Save processed datasets
X_train_scaled.to_csv("processed_regression/X_train.csv",index=False)

X_test_scaled.to_csv("processed_regression/X_test.csv",index=False)

y_train.to_csv("processed_regression/y_train.csv",index=False)

y_test.to_csv("processed_regression/y_test.csv",index=False)


In [26]:
# Save preprocessing objects
joblib.dump(imputer,"processed_regression/imputer.pkl")

joblib.dump(ohe,"processed_regression/onehot_encoder.pkl")

joblib.dump(scaler,"processed_regression/scaler.pkl")

['processed_regression/scaler.pkl']

In [27]:
# 10. FINAL CHECK
# ============================================================

print("\n==========================================")
print("[SUCCESS] Regression preprocessing complete!")
print("==========================================")

print("\nFinal data:")
print("X_train:", X_train_scaled.shape)
print("X_test :", X_test_scaled.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

print("\nSaved files:")
print(" - X_train.csv")
print(" - X_test.csv")
print(" - y_train.csv")
print(" - y_test.csv")
print(" - imputer.pkl")
print(" - onehot_encoder.pkl")
print(" - scaler.pkl")

print("\nTarget transformation: log1p(diameter)")


[SUCCESS] Regression preprocessing complete!

Final data:
X_train: (108967, 22)
X_test : (27242, 22)
y_train: (108967,)
y_test : (27242,)

Saved files:
 - X_train.csv
 - X_test.csv
 - y_train.csv
 - y_test.csv
 - imputer.pkl
 - onehot_encoder.pkl
 - scaler.pkl

Target transformation: log1p(diameter)
